# 05 — SIMCA mixture application

Objectif : appliquer aux images mixtures les configurations SIMCA figées en 04C.

Ce notebook ne fait plus de sélection de modèle ni de calibration de seuils :
- les modèles viennent de `frozen_reference_configs.parquet` ;
- les seuils 2-way `object_threshold` sont figés ;
- les seuils 3-way `three_way_lower_threshold` et `three_way_upper_threshold` sont figés ;
- les images mixtures servent uniquement à l’application finale et à l’interprétation.

Sorties principales :
- prédictions objet et pixel sur mixtures ;
- métriques objet/pixel sur mixtures ;
- décisions objet 3-way fixes ;
- confusion tables 3-way avec confiance ;
- cartes de décision 3-way objectwise ;
- diagnostics SIMCA Q residuals vs Hotelling T².

In [ ]:
from __future__ import annotations

import sys
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 260)
pd.set_option("display.max_rows", 300)

CURRENT_DIR = Path.cwd().resolve()

if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        "Could not find project root. Run this notebook from the project root or notebooks/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
from src.io.database_h5 import load_nir_uco_h5

from src.utils import (
    save_parquet,
    save_parquet_if_nonempty,
    load_parquet,
    list_result_files,
)

from src.spectra.band_selection import (
    select_wavelength_range_from_database,
    wavelength_selection_summary,
)

from src.spectra.preprocessing_configs import normalize_preprocessing_configs

from src.decision.labels import (
    predicted_col,
    true_col,
    pixel_ratio_col,
    UNCERTAIN_LABEL,
)

from src.decision.metrics import (
    add_binary_confusion_case,
    summarize_object_errors_by_image,
    summarize_pixel_errors_by_image,
)

from src.decision.aggregation import (
    object_threshold_grid,
)

from src.decision.truth import (
    add_pixel_truth_labels,
)

from src.decision.uncertainty import (
    add_three_way_object_decision,
    evaluate_three_way_by_config,
    add_three_way_confidence,
    three_way_confusion_table,
)

from src.decision.border import (
    summarize_border_diagnostics_by_config,
)

from src.workflows.simca import (
    make_target_train_filters,
    refit_selected_simca_configs,
)

from src.workflows.simca_selection_utils import (
    ensure_candidate_columns,
    normalize_simca_rule_columns,
    add_detection_selection_score,
    sort_detection_selection,
    add_reference_selection_scores,
    fill_selected_config_defaults,
    summarize_parameter_tendencies,
)


from src.visualization.plot_decision import (
    plot_object_decision_map,
    plot_object_error_overlay,
    plot_object_fp_fn_overlay,
    plot_pixel_prediction_overlay,
    plot_pixel_error_overlay,
    plot_pixel_fp_fn_overlay,
    plot_pixel_three_way_decision_overlay,
    plot_three_way_confusion_heatmap,
    plot_binary_confusion_heatmap,
)
from src.visualization.plot_simca import (
    plot_simca_q_t2_dataframe,
)

%load_ext autoreload
%autoreload 2

## 1. Configuration

In [ ]:
# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
DB_H5_PATH = PROJECT_ROOT / "HSI Data" / "processed" / "nir_uco_database.h5"

# ---------------------------------------------------------------------
# Spectral configuration
# ---------------------------------------------------------------------
USE_WAVELENGTH_WINDOW = False
WAVELENGTH_MODE = "non_noisy_all"

WINDOW_MIN_NM = 1225.0
WINDOW_MAX_NM = 1675.0

if USE_WAVELENGTH_WINDOW:
    RESULTS_TAG = f"{int(WINDOW_MIN_NM)}_{int(WINDOW_MAX_NM)}"
else:
    RESULTS_TAG = "non_noisy_all"

# ---------------------------------------------------------------------
# Inputs from 04C
# ---------------------------------------------------------------------
RESULTS_04C_DIR = PROJECT_ROOT / "results" / f"04C_simca_pure_test_{RESULTS_TAG}"

FROZEN_REFERENCE_CONFIGS_PATH = (
    RESULTS_04C_DIR / "frozen_reference_configs.parquet"
)

PURE_TEST_METRICS_PATH = (
    RESULTS_04C_DIR / "pure_test_metrics.parquet"
)

# ---------------------------------------------------------------------
# Outputs
# ---------------------------------------------------------------------
RESULTS_DIR = PROJECT_ROOT / "results" / f"05_simca_mixture_application_{RESULTS_TAG}"
DEBUG_DIR = RESULTS_DIR / "debug"
FIGURES_DIR = RESULTS_DIR / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

MIXTURE_METRICS_PATH = RESULTS_DIR / "mixture_metrics.parquet"
MIXTURE_MODEL_SUMMARY_PATH = RESULTS_DIR / "mixture_model_summary.parquet"

MIXTURE_OBJECT_PREDICTIONS_PATH = RESULTS_DIR / "mixture_object_predictions.parquet"
MIXTURE_OBJECT_ERRORS_BY_IMAGE_PATH = RESULTS_DIR / "mixture_object_errors_by_image.parquet"
MIXTURE_PIXEL_ERRORS_BY_IMAGE_PATH = RESULTS_DIR / "mixture_pixel_errors_by_image.parquet"

MIXTURE_OBJECT_2WAY_CONFUSION_PATH = RESULTS_DIR / "mixture_object_2way_confusion.parquet"
MIXTURE_PIXEL_2WAY_CONFUSION_PATH = RESULTS_DIR / "mixture_pixel_2way_confusion.parquet"

MIXTURE_THREE_WAY_METRICS_PATH = RESULTS_DIR / "mixture_three_way_metrics.parquet"
MIXTURE_THREE_WAY_BY_IMAGE_PATH = RESULTS_DIR / "mixture_three_way_by_image.parquet"
MIXTURE_OBJECT_PREDICTIONS_3WAY_PATH = RESULTS_DIR / "mixture_object_predictions_3way.parquet"
MIXTURE_PIXEL_PREDICTIONS_3WAY_PATH = RESULTS_DIR / "mixture_pixel_predictions_3way_from_object_decision.parquet"

MIXTURE_OBJECT_3WAY_CONFUSION_PATH = RESULTS_DIR / "mixture_object_3way_confusion.parquet"
MIXTURE_PIXEL_3WAY_CONFUSION_PATH = RESULTS_DIR / "mixture_pixel_3way_confusion_from_object_decision.parquet"

MIXTURE_OBJECT_SIMCA_DIAGNOSTICS_PATH = RESULTS_DIR / "mixture_object_simca_q_t2_diagnostics.parquet"
MIXTURE_PIXEL_SIMCA_DIAGNOSTICS_PATH = RESULTS_DIR / "mixture_pixel_simca_q_t2_diagnostics.parquet"

MIXTURE_BORDER_DIAGNOSTIC_PATH = RESULTS_DIR / "mixture_border_diagnostic.parquet"
MIXTURE_TRUTH_DILATION_SENSITIVITY_PATH = RESULTS_DIR / "mixture_truth_dilation_sensitivity.parquet"

MIXTURE_PARAMETER_TENDENCIES_PATH = RESULTS_DIR / "mixture_parameter_tendencies.parquet"
MIXTURE_APPLICATION_PROTOCOL_PATH = RESULTS_DIR / "mixture_application_protocol.parquet"
MIXTURE_REFIT_ERRORS_PATH = RESULTS_DIR / "mixture_refit_errors.parquet"

MIXTURE_PIXEL_PREDICTIONS_MINIMAL_PATH = (
    DEBUG_DIR / "mixture_pixel_predictions_minimal.parquet"
)

# ---------------------------------------------------------------------
# Detection protocol
# ---------------------------------------------------------------------
TARGET_CLASS = "peanut"
NON_TARGET_LABEL = "almond"
UNCERTAIN_LABEL_USED = UNCERTAIN_LABEL

REFERENCE_CLASSES = ("almond", TARGET_CLASS)

# Final calibration for application:
# the model configurations were frozen in 04C, so all pure peanut batches
# can now be used to calibrate the final models before projecting mixtures.
MIXTURE_FINAL_TRAIN_BATCHES = [1, 2, 3, 4]

MIXTURE_FINAL_TRAIN_FILTERS = make_target_train_filters(
    target_class=TARGET_CLASS,
    train_batches=MIXTURE_FINAL_TRAIN_BATCHES,
)

MIXTURE_FILTERS = {
    "sample_kind": ["mixture"],
}

# ---------------------------------------------------------------------
# Runtime flags
# ---------------------------------------------------------------------
RUN_MIXTURE_APPLICATION = True
APPLY_FIXED_THREE_WAY = True
RUN_BORDER_DIAGNOSTIC = True
RUN_TRUTH_DILATION_SENSITIVITY = True
RUN_VISUALIZATION = True
RUN_TWO_WAY_VISUALIZATION = True
RUN_THREE_WAY_VISUALIZATION = True

SAVE_MIXTURE_PIXEL_TABLES = False

RANDOM_STATE = 42
REPLACE_BALANCED_PIXELS = False

CV_N_SPLITS = 5
CV_GROUP_COL = "object_id"

# Limit number of frozen models applied to mixtures if needed.
# Use None to apply all frozen models.
MAX_REFERENCE_CONFIGS = None

# Border/core diagnostic.
BORDER_DIAGNOSTIC_WIDTHS = [1, 2, 3]

# Mixture truth sensitivity.
# Prediction is fixed; only the approximate mixture truth map is recomputed.
TRUTH_DILATION_RADII = [0, 1, 2, 3, 4, 5]

# Visualization.
N_CONFIGS_TO_VISUALIZE_PER_FAMILY = 2
N_IMAGES_PER_CONFIG = 3
IMAGE_SELECTION_MODE = "best"  # "worst" or "best"
SAVE_FIGURES_HTML = True
MAX_PIXELS_FOR_QT2_PLOT = 8000

print("DB_H5_PATH:", DB_H5_PATH)
print("RESULTS_04C_DIR:", RESULTS_04C_DIR)
print("FROZEN_REFERENCE_CONFIGS_PATH:", FROZEN_REFERENCE_CONFIGS_PATH)
print("RESULTS_DIR:", RESULTS_DIR)
print("MIXTURE_FINAL_TRAIN_FILTERS:", MIXTURE_FINAL_TRAIN_FILTERS)
print("MIXTURE_FILTERS:", MIXTURE_FILTERS)

## 2. Load database and frozen configurations

In [ ]:
if not DB_H5_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_H5_PATH}. Run notebook 00 first.")

if not FROZEN_REFERENCE_CONFIGS_PATH.exists():
    raise FileNotFoundError(
        f"Frozen reference configs not found: {FROZEN_REFERENCE_CONFIGS_PATH}. "
        "Run notebook 04C first."
    )

object_db, image_db = load_nir_uco_h5(
    DB_H5_PATH,
    reconstruct_heavy_object_arrays=True,
)

# ---------------------------------------------------------------------
# Wavelength handling
# ---------------------------------------------------------------------
if USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, wavelength_info = select_wavelength_range_from_database(
        object_db=object_db,
        image_db=image_db,
        min_nm=WINDOW_MIN_NM,
        max_nm=WINDOW_MAX_NM,
    )

    wavelength_selection_df = wavelength_selection_summary(wavelength_info)

else:
    first_obj = next(iter(object_db.values()))
    wavelengths = first_obj.get("wavelengths")
    wavelengths = np.asarray(wavelengths, dtype=float) if wavelengths is not None else None
    wavelength_selection_df = pd.DataFrame()

if wavelengths is None:
    raise RuntimeError("No wavelength axis found in object_db.")

wavelength_config_df = pd.DataFrame([{
    "wavelength_mode": WAVELENGTH_MODE,
    "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
    "results_tag": RESULTS_TAG,
    "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "n_bands": int(len(wavelengths)),
    "min_wavelength_nm": float(np.min(wavelengths)),
    "max_wavelength_nm": float(np.max(wavelengths)),
}])

reference_configs_df = load_parquet(FROZEN_REFERENCE_CONFIGS_PATH)

if PURE_TEST_METRICS_PATH.exists():
    pure_test_metrics_df = load_parquet(PURE_TEST_METRICS_PATH)
else:
    pure_test_metrics_df = pd.DataFrame()

reference_configs_df = ensure_candidate_columns(reference_configs_df)
reference_configs_df = normalize_simca_rule_columns(reference_configs_df)
reference_configs_df = fill_selected_config_defaults(
    reference_configs_df,
    default_values={
        "target_class": TARGET_CLASS,
        "non_target_label": NON_TARGET_LABEL,
        "sg_window_length": 11,
        "sg_polyorder": 2,
        "position_dilation_radius": 3,
        "m": 40,
        "alpha": 0.05,
        "object_threshold": 0.75,
    },
)
reference_configs_df = reference_configs_df.copy()
reference_configs_df["selected_config_id"] = reference_configs_df["selected_config_id"].astype(str)

# Les modèles sont déjà figés en 04C.
# On préserve donc leur ordre et leur rang de sélection.
if "frozen_reference_rank" in reference_configs_df.columns:
    reference_configs_df = (
        reference_configs_df
        .sort_values("frozen_reference_rank")
        .reset_index(drop=True)
    )
else:
    reference_configs_df["frozen_reference_rank"] = np.arange(
        1,
        len(reference_configs_df) + 1,
    )

# Ne garder que les modèles déclarés prêts pour l'application mixture.
if "ready_for_mixture_application" in reference_configs_df.columns:
    reference_configs_df = reference_configs_df[
        reference_configs_df["ready_for_mixture_application"].astype(bool)
    ].copy()

required_threshold_cols = [
    "selected_config_id",
    "object_threshold",
    "three_way_lower_threshold",
    "three_way_upper_threshold",
]

missing_threshold_cols = [
    col for col in required_threshold_cols
    if col not in reference_configs_df.columns
]

if missing_threshold_cols:
    raise KeyError(
        "Missing fixed decision thresholds in frozen_reference_configs_df: "
        f"{missing_threshold_cols}. Re-run 04C with fixed 3-way threshold export."
    )

if MAX_REFERENCE_CONFIGS is not None:
    reference_configs_df = (
        reference_configs_df
        .head(int(MAX_REFERENCE_CONFIGS))
        .copy()
        .reset_index(drop=True)
    )

print("Database loaded")
print("n objects:", len(object_db))
print("n images:", len(image_db))
print("n active bands:", len(wavelengths))
print("Frozen reference configs:", reference_configs_df.shape)
print("Pure test metrics:", pure_test_metrics_df.shape)

display(wavelength_config_df)
display(reference_configs_df.head(30))

In [ ]:
def _parse_preprocessing_steps(value):
    if isinstance(value, (list, tuple)):
        return tuple(str(v) for v in value)

    value = str(value)

    if "+" in value:
        return tuple(v.strip() for v in value.split("+") if v.strip())

    return (value.strip(),)


PREPROCESSING_CONFIGS = {
    str(row["preprocessing"]): _parse_preprocessing_steps(row["preprocessing_steps"])
    for _, row in reference_configs_df.drop_duplicates("preprocessing").iterrows()
}

PREPROCESSING_CONFIGS = normalize_preprocessing_configs(PREPROCESSING_CONFIGS)

print("Preprocessing configs used for final mixture refit:")
display(
    pd.DataFrame(
        [
            {
                "preprocessing": name,
                "preprocessing_steps": "+".join(steps),
            }
            for name, steps in PREPROCESSING_CONFIGS.items()
        ]
    )
)

## 3. Apply frozen SIMCA reference models to mixture images

In [ ]:
def enrich_with_reference_metadata(
    df: pd.DataFrame,
    reference_df: pd.DataFrame,
    id_col: str = "selected_config_id",
) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame() if df is None else df.copy()

    if id_col not in df.columns:
        return df.copy()

    metadata_cols = [
        "candidate_source",
        "candidate_input_rank",
        "frozen_reference_rank",
        "ready_for_mixture_application",

        "selection_split",
        "selection_strategy",

        "three_way_lower_threshold",
        "three_way_upper_threshold",

        "validation_fn_rate",
        "validation_fp_rate",
        "validation_3way_target_miss_rate",
        "validation_3way_non_target_false_accept_rate",
        "validation_3way_uncertain_rate",
        "validation_3way_coverage_rate",

        "pure_test_fn_rate",
        "pure_test_fp_rate",
        "pure_test_3way_target_miss_rate",
        "pure_test_3way_non_target_false_accept_rate",
        "pure_test_3way_uncertain_rate",
        "pure_test_3way_coverage_rate",

        "passes_pure_test_guardrail",
        "pure_test_guardrail_reason",
    ]

    metadata_cols = [
        col for col in metadata_cols
        if col in reference_df.columns and col not in df.columns
    ]

    if not metadata_cols:
        return df.copy()

    meta = (
        reference_df[[id_col] + metadata_cols]
        .drop_duplicates(id_col)
        .copy()
    )

    return df.merge(meta, on=id_col, how="left")

In [ ]:
if not RUN_MIXTURE_APPLICATION:
    raise RuntimeError(
        "RUN_MIXTURE_APPLICATION=False is not recommended because intermediate "
        "pixel/object tables are not saved by default."
    )

(
    mixture_metrics_df,
    mixture_objects_df,
    mixture_pixels_df,
    mixture_pixel_errors_by_image_df,
    mixture_refit_errors_df,
) = refit_selected_simca_configs(
    selected_configs_df=reference_configs_df,
    object_db=object_db,
    image_db=image_db,
    train_filters=MIXTURE_FINAL_TRAIN_FILTERS,
    projection_filters=MIXTURE_FILTERS,
    preprocessing_configs=PREPROCESSING_CONFIGS,
    evaluation_split="mixture_application",
    wavelengths=wavelengths,
    random_state=RANDOM_STATE,
    replace=REPLACE_BALANCED_PIXELS,
    cv_n_splits=CV_N_SPLITS,
    cv_group_col=CV_GROUP_COL,
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
)

mixture_objects_df = add_binary_confusion_case(
    mixture_objects_df,
    target_class=TARGET_CLASS,
    level="object",
)

mixture_pixels_df = add_binary_confusion_case(
    mixture_pixels_df,
    target_class=TARGET_CLASS,
    level="pixel",
)

mixture_objects_df = enrich_with_reference_metadata(
    mixture_objects_df,
    reference_configs_df,
)

mixture_pixels_df = enrich_with_reference_metadata(
    mixture_pixels_df,
    reference_configs_df,
)

save_parquet(mixture_metrics_df, MIXTURE_METRICS_PATH)
save_parquet_if_nonempty(mixture_refit_errors_df, MIXTURE_REFIT_ERRORS_PATH)

print("Mixture metrics:", mixture_metrics_df.shape)
print("Mixture objects:", mixture_objects_df.shape)
print("Mixture pixels:", mixture_pixels_df.shape)
print("Refit errors:", mixture_refit_errors_df.shape)
print("Saved:", MIXTURE_METRICS_PATH)

display(mixture_metrics_df.head(30))
display(mixture_refit_errors_df)

## 4. Two-way performance summaries on mixture images

In [ ]:
MODEL_GROUP_COLS = [
    "selected_config_id",
    "selection_strategy",
    "matrix_family",
    "training_matrix_id",
    "model_family",
    "matrix_method",
    "preprocessing",
    "selected_rule_name",
    "rule",
    "rule_variant",
    "rule_for_refit",
    "n_components",
    "alpha",
    "object_threshold",
    "sg_window_length",
    "sg_polyorder",
    "position_dilation_radius",
    "m",
    "m_effective",
    "balanced_pixel_strategy",
    "balanced_pixel_strategy_effective",
]

MODEL_GROUP_COLS = [
    col for col in MODEL_GROUP_COLS
    if col in mixture_objects_df.columns
    and col in mixture_pixels_df.columns
]

mixture_object_by_model_df = summarize_object_errors_by_image(
    mixture_objects_df,
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
    group_cols=MODEL_GROUP_COLS,
)

mixture_pixel_by_model_df = summarize_pixel_errors_by_image(
    mixture_pixels_df,
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
    group_cols=MODEL_GROUP_COLS,
)

print("Object metrics by model:", mixture_object_by_model_df.shape)
print("Pixel metrics by model:", mixture_pixel_by_model_df.shape)

display(
    mixture_object_by_model_df
    .sort_values(["fn_rate", "fp_rate", "balanced_accuracy"], ascending=[True, True, False])
    .head(30)
)

display(
    mixture_pixel_by_model_df
    .sort_values(["fn_rate", "fp_rate", "balanced_accuracy"], ascending=[True, True, False])
    .head(30)
)

In [ ]:
KEY_COLS = [
    col for col in MODEL_GROUP_COLS
    if col in mixture_object_by_model_df.columns
    and col in mixture_pixel_by_model_df.columns
]

GENERIC_METRIC_COLS = [
    "n",
    "tp",
    "fn",
    "fp",
    "tn",
    "target_sensitivity",
    "non_target_specificity",
    "balanced_accuracy",
    "accuracy",
    "precision",
    "f1_score",
    "fn_rate",
    "fp_rate",
]

object_metric_cols = [
    col for col in GENERIC_METRIC_COLS
    if col in mixture_object_by_model_df.columns
]

pixel_metric_cols = [
    col for col in GENERIC_METRIC_COLS
    if col in mixture_pixel_by_model_df.columns
]

object_prefixed = (
    mixture_object_by_model_df[KEY_COLS + object_metric_cols]
    .rename(columns={col: f"object_{col}" for col in object_metric_cols})
)

pixel_prefixed = (
    mixture_pixel_by_model_df[KEY_COLS + pixel_metric_cols]
    .rename(columns={col: f"pixel_{col}" for col in pixel_metric_cols})
)

mixture_model_summary_df = object_prefixed.merge(
    pixel_prefixed,
    on=KEY_COLS,
    how="outer",
)

# Diagnostic only: this score helps inspect mixture behaviour, but it is not used
# to select or re-rank the final reference models.
mixture_model_summary_df["mixture_reference_score"] = (
    -20.0 * mixture_model_summary_df["object_fn_rate"].fillna(1.0)
    -3.0 * mixture_model_summary_df["object_fp_rate"].fillna(1.0)
    -2.0 * mixture_model_summary_df["pixel_fn_rate"].fillna(1.0)
    -1.0 * mixture_model_summary_df["pixel_fp_rate"].fillna(1.0)
    +2.0 * mixture_model_summary_df["object_balanced_accuracy"].fillna(0.0)
    +0.5 * mixture_model_summary_df["pixel_balanced_accuracy"].fillna(0.0)
)

sort_cols = [
    "object_fn_rate",
    "pixel_fn_rate",
    "object_fp_rate",
    "pixel_fp_rate",
    "mixture_reference_score",
]

sort_cols = [col for col in sort_cols if col in mixture_model_summary_df.columns]

ascending = [
    True if col != "mixture_reference_score" else False
    for col in sort_cols
]

mixture_model_summary_df = (
    mixture_model_summary_df
    .sort_values(sort_cols, ascending=ascending)
    .reset_index(drop=True)
)

save_parquet(mixture_model_summary_df, MIXTURE_MODEL_SUMMARY_PATH)

print("Mixture model summary:", mixture_model_summary_df.shape)
print("Saved:", MIXTURE_MODEL_SUMMARY_PATH)

display(
    mixture_model_summary_df[
        [
            col for col in [
                "selected_config_id",
                "matrix_family",
                "training_matrix_id",
                "model_family",
                "matrix_method",
                "preprocessing",
                "selected_rule_name",
                "n_components",
                "alpha",
                "object_threshold",
                "object_fn",
                "object_fp",
                "object_balanced_accuracy",
                "object_fn_rate",
                "object_fp_rate",
                "pixel_fn",
                "pixel_fp",
                "pixel_balanced_accuracy",
                "pixel_fn_rate",
                "pixel_fp_rate",
                "mixture_reference_score",
            ]
            if col in mixture_model_summary_df.columns
        ]
    ].head(40)
)

In [ ]:
IMAGE_GROUP_COLS_OBJECT = MODEL_GROUP_COLS + ["source_image"]
IMAGE_GROUP_COLS_OBJECT = [
    col for col in IMAGE_GROUP_COLS_OBJECT
    if col in mixture_objects_df.columns
]

IMAGE_GROUP_COLS_PIXEL = MODEL_GROUP_COLS + ["source_image"]
IMAGE_GROUP_COLS_PIXEL = [
    col for col in IMAGE_GROUP_COLS_PIXEL
    if col in mixture_pixels_df.columns
]

mixture_object_errors_by_image_df = summarize_object_errors_by_image(
    mixture_objects_df,
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
    group_cols=IMAGE_GROUP_COLS_OBJECT,
)

mixture_pixel_errors_by_image_df = summarize_pixel_errors_by_image(
    mixture_pixels_df,
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
    group_cols=IMAGE_GROUP_COLS_PIXEL,
)

save_parquet(mixture_object_errors_by_image_df, MIXTURE_OBJECT_ERRORS_BY_IMAGE_PATH)
save_parquet(mixture_pixel_errors_by_image_df, MIXTURE_PIXEL_ERRORS_BY_IMAGE_PATH)

print("Object errors by image:", mixture_object_errors_by_image_df.shape)
print("Pixel errors by image:", mixture_pixel_errors_by_image_df.shape)
print("Saved:")
print(" -", MIXTURE_OBJECT_ERRORS_BY_IMAGE_PATH)
print(" -", MIXTURE_PIXEL_ERRORS_BY_IMAGE_PATH)

display(mixture_object_errors_by_image_df.head(30))
display(mixture_pixel_errors_by_image_df.head(30))

## 4B. Two-way projection tables and confidence

In [ ]:
# 2-way confidence and confusion tables
# ---------------------------------------------------------------------
# These tables are the 2-way counterpart of the requested 3-way figures:
# - object-level confusion table with confidence
# - pixel-level confusion table with confidence
#
# Confidence is only a diagnostic:
# - object level: distance from the object_threshold in target-pixel-ratio space
# - pixel level: distance from the SIMCA rule limit when rule_statistic/rule_limit exist

def add_binary_object_confidence(
    object_df: pd.DataFrame,
    target_class: str = TARGET_CLASS,
    ratio_col: str | None = None,
    threshold_col: str = "object_threshold",
    output_margin_col: str = "binary_margin",
    output_confidence_col: str = "binary_confidence",
    eps: float = 1e-12,
) -> pd.DataFrame:
    if object_df is None or len(object_df) == 0:
        return pd.DataFrame() if object_df is None else object_df.copy()

    if ratio_col is None:
        ratio_col = pixel_ratio_col(target_class)

    df = object_df.copy()

    if ratio_col not in df.columns or threshold_col not in df.columns:
        df[output_margin_col] = np.nan
        df[output_confidence_col] = np.nan
        return df

    ratio = pd.to_numeric(df[ratio_col], errors="coerce")
    threshold = pd.to_numeric(df[threshold_col], errors="coerce")

    margin = (ratio - threshold).abs()
    denom = np.where(
        ratio >= threshold,
        (1.0 - threshold).clip(lower=eps),
        threshold.clip(lower=eps),
    )

    df[output_margin_col] = margin
    df[output_confidence_col] = (margin / denom).clip(lower=0.0, upper=1.0)

    df["binary_confidence_bin"] = pd.cut(
        df[output_confidence_col],
        bins=[-np.inf, 0.33, 0.66, np.inf],
        labels=["low", "medium", "high"],
    ).astype("object")

    return df


def add_binary_pixel_confidence(
    pixel_df: pd.DataFrame,
    output_margin_col: str = "binary_margin",
    output_confidence_col: str = "binary_confidence",
    eps: float = 1e-12,
) -> pd.DataFrame:
    if pixel_df is None or len(pixel_df) == 0:
        return pd.DataFrame() if pixel_df is None else pixel_df.copy()

    df = pixel_df.copy()

    if {"rule_statistic", "rule_limit"}.issubset(df.columns):
        stat = pd.to_numeric(df["rule_statistic"], errors="coerce")
        limit = pd.to_numeric(df["rule_limit"], errors="coerce").clip(lower=eps)
        ratio = stat / limit

        df["rule_statistic_norm_limit"] = ratio
        df[output_margin_col] = (ratio - 1.0).abs()
        df[output_confidence_col] = df[output_margin_col].clip(lower=0.0, upper=1.0)

    else:
        df[output_margin_col] = np.nan
        df[output_confidence_col] = np.nan

    df["binary_confidence_bin"] = pd.cut(
        df[output_confidence_col],
        bins=[-np.inf, 0.33, 0.66, np.inf],
        labels=["low", "medium", "high"],
    ).astype("object")

    return df


def binary_confusion_table(
    df: pd.DataFrame,
    true_col_name: str,
    pred_col_name: str,
    confidence_col: str | None = "binary_confidence",
    group_cols=(),
    target_class: str = TARGET_CLASS,
    non_target_label: str = NON_TARGET_LABEL,
) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame()

    group_cols = [col for col in list(group_cols) if col in df.columns]

    required = [true_col_name, pred_col_name]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise KeyError(f"Missing columns for 2-way confusion table: {missing}")

    d = df[df[true_col_name].notna() & df[pred_col_name].notna()].copy()
    if len(d) == 0:
        return pd.DataFrame()

    d["true_label_2way"] = np.where(
        d[true_col_name].astype(bool),
        target_class,
        non_target_label,
    )
    d["predicted_label_2way"] = np.where(
        d[pred_col_name].astype(bool),
        target_class,
        non_target_label,
    )

    rows = []
    labels = [non_target_label, target_class]

    grouped = d.groupby(group_cols, dropna=False) if group_cols else [((), d)]
    for key, group in grouped:
        if not isinstance(key, tuple):
            key = (key,)

        base = {col: value for col, value in zip(group_cols, key)}
        n_group = len(group)

        for true_label in labels:
            true_group = group[group["true_label_2way"].astype(str).eq(str(true_label))]
            n_true = len(true_group)

            for pred_label in labels:
                cell = true_group[
                    true_group["predicted_label_2way"].astype(str).eq(str(pred_label))
                ]

                row = dict(base)
                row.update(
                    {
                        "true_label_2way": true_label,
                        "predicted_label_2way": pred_label,
                        "n": int(len(cell)),
                        "n_true_label": int(n_true),
                        "n_group": int(n_group),
                        "row_rate": len(cell) / max(n_true, 1),
                        "global_rate": len(cell) / max(n_group, 1),
                    }
                )

                if confidence_col is not None and confidence_col in cell.columns:
                    conf = pd.to_numeric(cell[confidence_col], errors="coerce")
                    row["mean_confidence"] = float(conf.mean()) if conf.notna().any() else np.nan
                    row["median_confidence"] = float(conf.median()) if conf.notna().any() else np.nan
                else:
                    row["mean_confidence"] = np.nan
                    row["median_confidence"] = np.nan

                rows.append(row)

    return pd.DataFrame(rows)


mixture_objects_df = add_binary_object_confidence(
    mixture_objects_df,
    target_class=TARGET_CLASS,
)

mixture_pixels_df = add_binary_pixel_confidence(
    mixture_pixels_df,
)

object_true_col = true_col(TARGET_CLASS, "object")
object_pred_col = predicted_col(TARGET_CLASS, "object")
pixel_true_col = true_col(TARGET_CLASS, "pixel")
pixel_pred_col = predicted_col(TARGET_CLASS, "pixel")

TWO_WAY_GROUP_COLS = [
    "selected_config_id",
    "matrix_family",
    "candidate_source",
]

mixture_object_2way_confusion_df = binary_confusion_table(
    df=mixture_objects_df,
    true_col_name=object_true_col,
    pred_col_name=object_pred_col,
    confidence_col="binary_confidence",
    group_cols=TWO_WAY_GROUP_COLS,
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
)

mixture_pixel_2way_confusion_df = binary_confusion_table(
    df=mixture_pixels_df,
    true_col_name=pixel_true_col,
    pred_col_name=pixel_pred_col,
    confidence_col="binary_confidence",
    group_cols=TWO_WAY_GROUP_COLS,
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
)

save_parquet_if_nonempty(
    mixture_object_2way_confusion_df,
    MIXTURE_OBJECT_2WAY_CONFUSION_PATH,
)

save_parquet_if_nonempty(
    mixture_pixel_2way_confusion_df,
    MIXTURE_PIXEL_2WAY_CONFUSION_PATH,
)

print("Object 2-way confusion:", mixture_object_2way_confusion_df.shape)
print("Pixel 2-way confusion:", mixture_pixel_2way_confusion_df.shape)
print("Saved:")
print(" -", MIXTURE_OBJECT_2WAY_CONFUSION_PATH)
print(" -", MIXTURE_PIXEL_2WAY_CONFUSION_PATH)

display(mixture_object_2way_confusion_df.head(30))
display(mixture_pixel_2way_confusion_df.head(30))


## 5. Optional 3-way object decision on mixture images

In [ ]:
# ---------------------------------------------------------------------
# Apply fixed 3-way object decisions on mixture images
# ---------------------------------------------------------------------
# No threshold calibration is performed here.
# Thresholds come from frozen_reference_configs.parquet.

if APPLY_FIXED_THREE_WAY:
    mixture_three_way_metrics_df, mixture_objects_3way_df = evaluate_three_way_by_config(
        object_df=mixture_objects_df,
        thresholds_df=reference_configs_df,
        config_id_col="selected_config_id",
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
    )

    mixture_three_way_by_image_df, _ = evaluate_three_way_by_config(
        object_df=mixture_objects_df,
        thresholds_df=reference_configs_df,
        config_id_col="selected_config_id",
        extra_group_cols=["source_image"],
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
    )

    mixture_objects_3way_df = add_three_way_confidence(
        mixture_objects_3way_df,
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
        uncertain_label=UNCERTAIN_LABEL_USED,
        decision_col="decision_3way",
    )

    mixture_objects_3way_df = enrich_with_reference_metadata(
        mixture_objects_3way_df,
        reference_configs_df,
    )

else:
    mixture_three_way_metrics_df = pd.DataFrame()
    mixture_three_way_by_image_df = pd.DataFrame()
    mixture_objects_3way_df = pd.DataFrame()

save_parquet_if_nonempty(
    mixture_three_way_metrics_df,
    MIXTURE_THREE_WAY_METRICS_PATH,
)

save_parquet_if_nonempty(
    mixture_three_way_by_image_df,
    MIXTURE_THREE_WAY_BY_IMAGE_PATH,
)

save_parquet_if_nonempty(
    mixture_objects_3way_df,
    MIXTURE_OBJECT_PREDICTIONS_3WAY_PATH,
)

print("Fixed 3-way metrics:", mixture_three_way_metrics_df.shape)
print("Fixed 3-way by image:", mixture_three_way_by_image_df.shape)
print("Mixture objects with fixed 3-way decisions:", mixture_objects_3way_df.shape)

display(
    mixture_three_way_metrics_df[
        [
            col for col in [
                "selected_config_id",
                "n",
                "n_target",
                "n_non_target",
                "target_miss_rate",
                "screening_sensitivity",
                "non_target_false_accept_rate",
                "non_target_auto_reject_rate",
                "uncertain_rate",
                "coverage_rate",
                "decided_balanced_accuracy",
            ]
            if col in mixture_three_way_metrics_df.columns
        ]
    ].head(30)
)

In [ ]:
# ---------------------------------------------------------------------
# Assign objectwise 3-way decisions to pixels
# ---------------------------------------------------------------------

def assign_object_three_way_decision_to_pixels(
    pixel_df: pd.DataFrame,
    object_3way_df: pd.DataFrame,
) -> pd.DataFrame:
    if pixel_df is None or len(pixel_df) == 0:
        return pd.DataFrame() if pixel_df is None else pixel_df.copy()

    if object_3way_df is None or len(object_3way_df) == 0:
        return pixel_df.copy()

    keys = [
        "selected_config_id",
        "source_image",
        "object_id",
    ]

    required = keys + [
        "decision_3way",
        "three_way_lower_threshold",
        "three_way_upper_threshold",
        "three_way_confidence",
        "three_way_margin",
    ]

    keep_cols = [
        col for col in required
        if col in object_3way_df.columns
    ]

    missing_keys = [
        col for col in keys
        if col not in pixel_df.columns or col not in object_3way_df.columns
    ]

    if missing_keys:
        raise KeyError(f"Missing keys for pixel/object 3-way merge: {missing_keys}")

    object_decisions = (
        object_3way_df[keep_cols]
        .drop_duplicates(keys)
        .copy()
    )

    out = pixel_df.merge(
        object_decisions,
        on=keys,
        how="left",
        suffixes=("", "_object_3way"),
    )

    return out


mixture_pixels_3way_df = assign_object_three_way_decision_to_pixels(
    pixel_df=mixture_pixels_df,
    object_3way_df=mixture_objects_3way_df,
)

save_parquet_if_nonempty(
    mixture_pixels_3way_df,
    MIXTURE_PIXEL_PREDICTIONS_3WAY_PATH,
)

print("Mixture pixels with objectwise 3-way decision:", mixture_pixels_3way_df.shape)
display(mixture_pixels_3way_df.head())

In [ ]:
# ---------------------------------------------------------------------
# 3-way confusion tables with confidence
# ---------------------------------------------------------------------

object_true_col = true_col(TARGET_CLASS, "object")
pixel_true_col = true_col(TARGET_CLASS, "pixel")

mixture_object_3way_confusion_df = three_way_confusion_table(
    df=mixture_objects_3way_df,
    true_col=object_true_col,
    decision_col="decision_3way",
    confidence_col="three_way_confidence",
    group_cols=[
        "selected_config_id",
        "matrix_family",
        "candidate_source",
    ],
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
    uncertain_label=UNCERTAIN_LABEL_USED,
)

mixture_pixel_3way_confusion_df = three_way_confusion_table(
    df=mixture_pixels_3way_df,
    true_col=pixel_true_col,
    decision_col="decision_3way",
    confidence_col="three_way_confidence",
    group_cols=[
        "selected_config_id",
        "matrix_family",
        "candidate_source",
    ],
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
    uncertain_label=UNCERTAIN_LABEL_USED,
)

save_parquet_if_nonempty(
    mixture_object_3way_confusion_df,
    MIXTURE_OBJECT_3WAY_CONFUSION_PATH,
)

save_parquet_if_nonempty(
    mixture_pixel_3way_confusion_df,
    MIXTURE_PIXEL_3WAY_CONFUSION_PATH,
)

print("Object 3-way confusion:", mixture_object_3way_confusion_df.shape)
print("Pixel 3-way confusion:", mixture_pixel_3way_confusion_df.shape)

display(mixture_object_3way_confusion_df.head(30))
display(mixture_pixel_3way_confusion_df.head(30))

In [ ]:
# ---------------------------------------------------------------------
# SIMCA Q residuals / Hotelling T² diagnostics
# ---------------------------------------------------------------------

SIMCA_DIAG_GROUP_KEYS = [
    "selected_config_id",
    "source_image",
    "object_id",
]

pixel_diag_cols = [
    "H",
    "Q",
    "H_norm_limit",
    "Q_norm_limit",
    "rule_statistic",
    "rule_limit",
]

available_pixel_diag_cols = [
    col for col in pixel_diag_cols
    if col in mixture_pixels_3way_df.columns
]

if available_pixel_diag_cols:
    object_simca_diag_df = (
        mixture_pixels_3way_df
        .groupby(SIMCA_DIAG_GROUP_KEYS, dropna=False)
        .agg(
            **{
                f"{col}_mean": (col, "mean")
                for col in available_pixel_diag_cols
            },
            **{
                f"{col}_median": (col, "median")
                for col in available_pixel_diag_cols
            },
            n_pixels_diag=("object_id", "size"),
        )
        .reset_index()
    )

    # Avoid duplicating diagnostic columns if the cell is rerun.
    cols_to_drop = [
        col for col in mixture_objects_3way_df.columns
        if col.endswith("_mean") or col.endswith("_median") or col == "n_pixels_diag"
    ]
    if cols_to_drop:
        mixture_objects_3way_df = mixture_objects_3way_df.drop(columns=cols_to_drop)

    mixture_objects_3way_df = mixture_objects_3way_df.merge(
        object_simca_diag_df,
        on=SIMCA_DIAG_GROUP_KEYS,
        how="left",
        suffixes=("", "_diag"),
    )

else:
    object_simca_diag_df = pd.DataFrame()

pixel_keep_cols = [
    col for col in [
        "selected_config_id",
        "source_image",
        "object_id",
        "row",
        "col",
        "label",
        "predicted_label_pixel",
        "decision_3way",
        "three_way_confidence",
        "H",
        "Q",
        "H_norm_limit",
        "Q_norm_limit",
        "rule_statistic",
        "rule_limit",
        "pixel_error_case",
    ]
    if col in mixture_pixels_3way_df.columns
]

object_keep_cols = [
    col for col in [
        "selected_config_id",
        "source_image",
        "object_id",
        "true_label_object",
        "predicted_label_object",
        "decision_3way",
        "three_way_confidence",
        "three_way_margin",
        "H_mean",
        "Q_mean",
        "H_norm_limit_mean",
        "Q_norm_limit_mean",
        "rule_statistic_mean",
        "rule_limit_mean",
        "object_error_case",
    ]
    if col in mixture_objects_3way_df.columns
]

mixture_pixel_simca_diagnostics_df = mixture_pixels_3way_df[pixel_keep_cols].copy()
mixture_object_simca_diagnostics_df = mixture_objects_3way_df[object_keep_cols].copy()

save_parquet_if_nonempty(
    mixture_pixel_simca_diagnostics_df,
    MIXTURE_PIXEL_SIMCA_DIAGNOSTICS_PATH,
)

save_parquet_if_nonempty(
    mixture_object_simca_diagnostics_df,
    MIXTURE_OBJECT_SIMCA_DIAGNOSTICS_PATH,
)

print("Pixel SIMCA diagnostics:", mixture_pixel_simca_diagnostics_df.shape)
print("Object SIMCA diagnostics:", mixture_object_simca_diagnostics_df.shape)

display(mixture_object_simca_diagnostics_df.head())
display(mixture_pixel_simca_diagnostics_df.head())

## 6. Save object predictions and optional minimal pixel table

In [ ]:
# Save object predictions after the 3-way analysis cell so that the table contains
# all core binary fields. The 3-way table is saved separately.
# Save the enriched object table with fixed 3-way decisions when available.
if len(mixture_objects_3way_df) > 0:
    save_parquet(mixture_objects_3way_df, MIXTURE_OBJECT_PREDICTIONS_PATH)
else:
    save_parquet(mixture_objects_df, MIXTURE_OBJECT_PREDICTIONS_PATH)
    
print("Saved object predictions:")
print(MIXTURE_OBJECT_PREDICTIONS_PATH)

if SAVE_MIXTURE_PIXEL_TABLES:
    if len(mixture_pixels_3way_df) > 0:
        save_parquet(
            mixture_pixels_3way_df,
            MIXTURE_PIXEL_PREDICTIONS_MINIMAL_PATH,
        )
    else:
        save_parquet(
            mixture_pixels_df,
            MIXTURE_PIXEL_PREDICTIONS_MINIMAL_PATH,
        )

In [ ]:
THREE_WAY_SUMMARY_COLS = [
    "selected_config_id",
    "n",
    "n_target",
    "n_non_target",
    "target_miss_rate",
    "screening_sensitivity",
    "non_target_false_accept_rate",
    "non_target_auto_reject_rate",
    "uncertain_rate",
    "coverage_rate",
    "decided_balanced_accuracy",
]

THREE_WAY_SUMMARY_COLS = [
    col for col in THREE_WAY_SUMMARY_COLS
    if col in mixture_three_way_metrics_df.columns
]

if THREE_WAY_SUMMARY_COLS:
    three_way_prefixed = (
        mixture_three_way_metrics_df[THREE_WAY_SUMMARY_COLS]
        .rename(
            columns={
                "n": "object_3way_n",
                "n_target": "object_3way_n_target",
                "n_non_target": "object_3way_n_non_target",
                "target_miss_rate": "object_3way_target_miss_rate",
                "screening_sensitivity": "object_3way_screening_sensitivity",
                "non_target_false_accept_rate": "object_3way_non_target_false_accept_rate",
                "non_target_auto_reject_rate": "object_3way_non_target_auto_reject_rate",
                "uncertain_rate": "object_3way_uncertain_rate",
                "coverage_rate": "object_3way_coverage_rate",
                "decided_balanced_accuracy": "object_3way_decided_balanced_accuracy",
            }
        )
    )

    # Make the cell rerunnable by dropping previous 3-way summary columns.
    three_way_cols_to_drop = [
        col for col in three_way_prefixed.columns
        if col != "selected_config_id" and col in mixture_model_summary_df.columns
    ]
    if three_way_cols_to_drop:
        mixture_model_summary_df = mixture_model_summary_df.drop(columns=three_way_cols_to_drop)

    mixture_model_summary_df = mixture_model_summary_df.merge(
        three_way_prefixed,
        on="selected_config_id",
        how="left",
    )

sort_cols = [
    "frozen_reference_rank",
    "matrix_family",
    "candidate_source",
    "selected_config_id",
]
sort_cols = [col for col in sort_cols if col in mixture_model_summary_df.columns]

mixture_model_summary_df = (
    mixture_model_summary_df
    .sort_values(sort_cols)
    .reset_index(drop=True)
)

save_parquet(mixture_model_summary_df, MIXTURE_MODEL_SUMMARY_PATH)

print("Updated mixture model summary with fixed 3-way metrics:", mixture_model_summary_df.shape)
print("Saved:", MIXTURE_MODEL_SUMMARY_PATH)

display(
    mixture_model_summary_df[
        [
            col for col in [
                "selected_config_id",
                "frozen_reference_rank",
                "candidate_source",
                "matrix_family",
                "training_matrix_id",
                "preprocessing",
                "selected_rule_name",
                "object_fn_rate",
                "object_fp_rate",
                "pixel_fn_rate",
                "pixel_fp_rate",
                "object_3way_target_miss_rate",
                "object_3way_non_target_false_accept_rate",
                "object_3way_uncertain_rate",
                "object_3way_coverage_rate",
            ]
            if col in mixture_model_summary_df.columns
        ]
    ].head(80)
)

## 7. Border/core diagnostics on mixture images

In [ ]:
if RUN_BORDER_DIAGNOSTIC:
    mixture_border_diagnostic_df = summarize_border_diagnostics_by_config(
        pixel_df=mixture_pixels_df,
        object_db=object_db,
        target_class=TARGET_CLASS,
        border_widths=BORDER_DIAGNOSTIC_WIDTHS,
        config_cols=[
            "selected_config_id",
            "matrix_family",
            "training_matrix_id",
            "matrix_method",
            "preprocessing",
            "selected_rule_name",
            "n_components",
            "alpha",
            "object_threshold",
        ],
    )
else:
    mixture_border_diagnostic_df = pd.DataFrame()

save_parquet_if_nonempty(
    mixture_border_diagnostic_df,
    MIXTURE_BORDER_DIAGNOSTIC_PATH,
)

print("Mixture border diagnostic:", mixture_border_diagnostic_df.shape)

display(mixture_border_diagnostic_df.head(80))

## 8. Sensitivity to mixture-truth dilation radius

In [ ]:
def evaluate_mixture_truth_dilation_sensitivity(
    pixel_df: pd.DataFrame,
    config_df: pd.DataFrame,
    image_db: dict,
    object_db: dict,
    target_class: str,
    non_target_label: str,
    dilation_radii,
) -> pd.DataFrame:
    """
    Recompute mixture truth maps with several dilation radii.

    Predictions are kept fixed. Only the approximate truth labels are changed.
    This helps determine whether model ranking depends strongly on the
    position-reference truth construction.
    """
    rows = []

    config_lookup = (
        config_df
        .drop_duplicates("selected_config_id")
        .set_index("selected_config_id")
        .to_dict(orient="index")
    )

    for dilation_radius in dilation_radii:
        print(f"[truth sensitivity] dilation_radius={dilation_radius}")

        pixel_truth_df = add_pixel_truth_labels(
            pixel_df=pixel_df,
            image_db=image_db,
            object_db=object_db,
            target_class=target_class,
            dilation_radius=int(dilation_radius),
        )

        for config_id, group in pixel_truth_df.groupby("selected_config_id", dropna=False):
            config_id = str(config_id)
            cfg = config_lookup.get(config_id, {})

            object_threshold = float(cfg.get("object_threshold", 0.75))

            threshold_df, _object_tables = object_threshold_grid(
                pixel_df=group,
                object_db=object_db,
                target_class=target_class,
                non_target_label=non_target_label,
                thresholds=[object_threshold],
            )

            if threshold_df is None or len(threshold_df) == 0:
                continue

            row = threshold_df.iloc[0].to_dict()
            row["selected_config_id"] = config_id
            row["truth_dilation_radius"] = int(dilation_radius)

            for col in [
                "matrix_family",
                "training_matrix_id",
                "matrix_method",
                "preprocessing",
                "selected_rule_name",
                "n_components",
                "alpha",
                "object_threshold",
            ]:
                if col in cfg:
                    row[col] = cfg[col]

            rows.append(row)

    if not rows:
        return pd.DataFrame()

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["selected_config_id", "truth_dilation_radius"],
            ascending=True,
        )
        .reset_index(drop=True)
    )


if RUN_TRUTH_DILATION_SENSITIVITY:
    mixture_truth_dilation_sensitivity_df = evaluate_mixture_truth_dilation_sensitivity(
        pixel_df=mixture_pixels_df,
        config_df=reference_configs_df,
        image_db=image_db,
        object_db=object_db,
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
        dilation_radii=TRUTH_DILATION_RADII,
    )
else:
    mixture_truth_dilation_sensitivity_df = pd.DataFrame()

save_parquet_if_nonempty(
    mixture_truth_dilation_sensitivity_df,
    MIXTURE_TRUTH_DILATION_SENSITIVITY_PATH,
)

print("Mixture truth dilation sensitivity:", mixture_truth_dilation_sensitivity_df.shape)

display(mixture_truth_dilation_sensitivity_df.head(80))

In [ ]:
if len(mixture_truth_dilation_sensitivity_df) > 0:
    truth_sensitivity_summary_df = (
        mixture_truth_dilation_sensitivity_df
        .groupby("selected_config_id", dropna=False)
        .agg(
            n_radii=("truth_dilation_radius", "nunique"),
            min_fn_rate=("fn_rate", "min"),
            max_fn_rate=("fn_rate", "max"),
            range_fn_rate=("fn_rate", lambda s: float(pd.to_numeric(s, errors="coerce").max() - pd.to_numeric(s, errors="coerce").min())),
            min_fp_rate=("fp_rate", "min"),
            max_fp_rate=("fp_rate", "max"),
            range_fp_rate=("fp_rate", lambda s: float(pd.to_numeric(s, errors="coerce").max() - pd.to_numeric(s, errors="coerce").min())),
            min_balanced_accuracy=("balanced_accuracy", "min"),
            max_balanced_accuracy=("balanced_accuracy", "max"),
            range_balanced_accuracy=("balanced_accuracy", lambda s: float(pd.to_numeric(s, errors="coerce").max() - pd.to_numeric(s, errors="coerce").min())),
        )
        .reset_index()
        .sort_values(["range_fn_rate", "range_fp_rate"], ascending=False)
    )

    print("Truth-sensitivity summary by model:")
    display(truth_sensitivity_summary_df.head(50))
else:
    truth_sensitivity_summary_df = pd.DataFrame()

## 9. Parameter tendencies among frozen reference models

In [ ]:
mixture_parameter_tendencies_df = summarize_parameter_tendencies(
    mixture_model_summary_df.rename(
        columns={
            "object_fn_rate": "fn_rate",
            "object_fp_rate": "fp_rate",
            "object_balanced_accuracy": "balanced_accuracy",
        }
    ),
    top_fraction=1.0,
    min_top_n=len(mixture_model_summary_df),
)

save_parquet_if_nonempty(
    mixture_parameter_tendencies_df,
    MIXTURE_PARAMETER_TENDENCIES_PATH,
)

print("Mixture parameter tendencies:", mixture_parameter_tendencies_df.shape)

display(mixture_parameter_tendencies_df.head(80))

## 10A. Visual diagnostics — 2-way projections

In [ ]:
# Visual diagnostics requested by tutors — 2-way projection section
# ---------------------------------------------------------------------
# This section mirrors the requested 3-way plots, but for the original
# binary SIMCA projection:
# 1) 2-way confusion table with confidence
# 2) objectwise 2-way decision map
# 3) SIMCA Q residuals vs Hotelling T², colored by 2-way decision

def choose_diagnostic_configs(reference_df, n_per_family=2):
    group_cols = ["matrix_family"]

    if "candidate_source" in reference_df.columns:
        group_cols.append("candidate_source")

    return (
        reference_df
        .sort_values("frozen_reference_rank")
        .groupby(group_cols, group_keys=False, dropna=False)
        .head(int(n_per_family))
        .reset_index(drop=True)
    )


def choose_images_for_config_2way(config_id, n_images=3, mode="worst"):
    d = mixture_object_errors_by_image_df[
        mixture_object_errors_by_image_df["selected_config_id"].astype(str).eq(str(config_id))
    ].copy()

    if len(d) == 0:
        return []

    sort_candidates = [
        "fn_rate",
        "fp_rate",
        "balanced_accuracy",
    ]
    sort_cols = [col for col in sort_candidates if col in d.columns]

    if not sort_cols:
        return (
            d["source_image"]
            .astype(str)
            .drop_duplicates()
            .head(int(n_images))
            .tolist()
        )

    if mode == "best":
        ascending = [
            True if col in {"fn_rate", "fp_rate"} else False
            for col in sort_cols
        ]
    else:
        ascending = [
            False if col in {"fn_rate", "fp_rate"} else True
            for col in sort_cols
        ]

    d = d.sort_values(sort_cols, ascending=ascending)

    return (
        d["source_image"]
        .astype(str)
        .drop_duplicates()
        .head(int(n_images))
        .tolist()
    )


def _sample_for_qt2_plot(df, label_col, max_rows=MAX_PIXELS_FOR_QT2_PLOT):
    if df is None or len(df) <= int(max_rows):
        return pd.DataFrame() if df is None else df.copy()

    if label_col in df.columns:
        n_groups = max(df[label_col].nunique(dropna=False), 1)
        n_per_group = max(1, int(max_rows) // n_groups)

        return (
            df.groupby(label_col, group_keys=False, dropna=False)
            .apply(
                lambda g: g.sample(
                    n=min(len(g), n_per_group),
                    random_state=RANDOM_STATE,
                )
            )
            .reset_index(drop=True)
        )

    return (
        df.sample(n=int(max_rows), random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )


diagnostic_configs_2way_df = choose_diagnostic_configs(
    reference_configs_df,
    n_per_family=N_CONFIGS_TO_VISUALIZE_PER_FAMILY,
)

if RUN_VISUALIZATION and RUN_TWO_WAY_VISUALIZATION:
    for _, cfg in diagnostic_configs_2way_df.iterrows():
        config_id = str(cfg["selected_config_id"])

        obj_2way_df = (
            mixture_objects_3way_df
            if isinstance(mixture_objects_3way_df, pd.DataFrame) and len(mixture_objects_3way_df) > 0
            else mixture_objects_df
        )
        obj_2way_df = obj_2way_df[
            obj_2way_df["selected_config_id"].astype(str).eq(config_id)
        ].copy()

        pix_2way_df = (
            mixture_pixels_3way_df
            if isinstance(mixture_pixels_3way_df, pd.DataFrame) and len(mixture_pixels_3way_df) > 0
            else mixture_pixels_df
        )
        pix_2way_df = pix_2way_df[
            pix_2way_df["selected_config_id"].astype(str).eq(config_id)
        ].copy()

        image_keys = choose_images_for_config_2way(
            config_id=config_id,
            n_images=N_IMAGES_PER_CONFIG,
            mode=IMAGE_SELECTION_MODE,
        )

        print("=" * 100)
        print(
            "2-way | "
            f"{config_id} | {cfg.get('candidate_source', '')} | "
            f"{cfg.get('matrix_family', '')} | {cfg.get('training_matrix_id', '')} | "
            f"{cfg.get('preprocessing', '')} | {cfg.get('selected_rule_name', '')}"
        )
        print("Images:", image_keys)

        # -------------------------------------------------------------
        # Figure 1 — 2-way confusion table with confidence
        # -------------------------------------------------------------
        conf_obj_2way = mixture_object_2way_confusion_df[
            mixture_object_2way_confusion_df["selected_config_id"].astype(str).eq(config_id)
        ].copy()

        if len(conf_obj_2way) > 0:
            try:
                fig = plot_binary_confusion_heatmap(
                    conf_obj_2way,
                    title=f"Object-level 2-way confusion — {config_id}",
                    show=True,
                )

                if SAVE_FIGURES_HTML and fig is not None:
                    fig.write_html(
                        FIGURES_DIR / f"{config_id}_object_2way_confusion.html"
                    )
            except Exception as exc:
                print(f"[WARNING] Object 2-way confusion plot failed for {config_id}: {exc!r}")
        else:
            print(f"[INFO] No object 2-way confusion table for {config_id}.")

        # -------------------------------------------------------------
        # Figure 2 — 2-way decision maps
        # -------------------------------------------------------------
        for image_key in image_keys:
            # if plot_object_decision_map is not None and len(obj_2way_df) > 0:
            #     try:
            #         fig = plot_object_decision_map(
            #             image_db=image_db,
            #             object_db=object_db,
            #             results_df=obj_2way_df,
            #             image_key=image_key,
            #             decision_col="predicted_label_object",
            #             decision_to_code={
            #                 NON_TARGET_LABEL: 1,
            #                 TARGET_CLASS: 2,
            #             },
            #             code_to_name={
            #                 1: NON_TARGET_LABEL,
            #                 2: TARGET_CLASS,
            #             },
            #             title=f"Objectwise 2-way decision — {config_id} — {image_key}",
            #             show=True,
            #         )

            #         if SAVE_FIGURES_HTML and fig is not None:
            #             fig.write_html(
            #                 FIGURES_DIR / f"{config_id}_{image_key}_objectwise_2way_map.html"
            #             )
            #     except Exception as exc:
            #         print(f"[WARNING] Objectwise 2-way map failed for {config_id}, image={image_key}: {exc!r}")

            # Optional object error overlay: TP/TN/FP/FN.
            if plot_object_error_overlay is not None and len(obj_2way_df) > 0:
                try:
                    fig = plot_object_error_overlay(
                        image_key=image_key,
                        image_db=image_db,
                        object_db=object_db,
                        object_df=obj_2way_df,
                        target_class=TARGET_CLASS,
                        title=f"Object-level 2-way errors — {config_id} — {image_key}",
                        show=True,
                    )

                    if SAVE_FIGURES_HTML and fig is not None:
                        fig.write_html(
                            FIGURES_DIR / f"{config_id}_{image_key}_object_2way_errors.html"
                        )
                except Exception as exc:
                    print(f"[WARNING] Object 2-way error overlay failed for {config_id}, image={image_key}: {exc!r}")

            # Pixel-level binary prediction map.
            if plot_pixel_prediction_overlay is not None and len(pix_2way_df) > 0:
                try:
                    fig = plot_pixel_prediction_overlay(
                        image_key=image_key,
                        image_db=image_db,
                        pixel_df=pix_2way_df,
                        target_class=TARGET_CLASS,
                        title=f"Pixel-level 2-way target prediction — {config_id} — {image_key}",
                        show=True,
                    )

                    if SAVE_FIGURES_HTML and fig is not None:
                        fig.write_html(
                            FIGURES_DIR / f"{config_id}_{image_key}_pixel_2way_prediction_map.html"
                        )
                except Exception as exc:
                    print(f"[WARNING] Pixel 2-way prediction map failed for {config_id}, image={image_key}: {exc!r}")

            # Optional pixel error overlay: TP/TN/FP/FN.
            if plot_pixel_error_overlay is not None and len(pix_2way_df) > 0:
                try:
                    fig = plot_pixel_error_overlay(
                        image_key=image_key,
                        image_db=image_db,
                        pixel_df=pix_2way_df,
                        target_class=TARGET_CLASS,
                        title=f"Pixel-level 2-way errors — {config_id} — {image_key}",
                        show=True,
                    )

                    if SAVE_FIGURES_HTML and fig is not None:
                        fig.write_html(
                            FIGURES_DIR / f"{config_id}_{image_key}_pixel_2way_errors.html"
                        )
                except Exception as exc:
                    print(f"[WARNING] Pixel 2-way error overlay failed for {config_id}, image={image_key}: {exc!r}")

        # -------------------------------------------------------------
        # Figure 3 — SIMCA Q residuals vs Hotelling T², colored by 2-way decision
        # -------------------------------------------------------------
        if plot_simca_q_t2_dataframe is not None:
            if len(obj_2way_df) > 0:
                try:
                    fig = plot_simca_q_t2_dataframe(
                        obj_2way_df,
                        level="object",
                        label_col="predicted_label_object",
                        confidence_col="binary_confidence",
                        title=f"Object-level SIMCA Q residuals vs Hotelling T² — 2-way — {config_id}",
                        show=True,
                    )

                    if SAVE_FIGURES_HTML and fig is not None:
                        fig.write_html(
                            FIGURES_DIR / f"{config_id}_object_qres_t2_2way.html"
                        )
                except Exception as exc:
                    print(f"[WARNING] Object 2-way Q/T² plot failed for {config_id}: {exc!r}")

            if len(pix_2way_df) > 0:
                pix_plot_df = _sample_for_qt2_plot(
                    pix_2way_df,
                    label_col="predicted_label_pixel",
                    max_rows=MAX_PIXELS_FOR_QT2_PLOT,
                )

                try:
                    fig = plot_simca_q_t2_dataframe(
                        pix_plot_df,
                        level="pixel",
                        label_col="predicted_label_pixel",
                        confidence_col="binary_confidence",
                        title=f"Pixel-level SIMCA Q residuals vs Hotelling T² — 2-way — {config_id}",
                        show=True,
                    )

                    if SAVE_FIGURES_HTML and fig is not None:
                        fig.write_html(
                            FIGURES_DIR / f"{config_id}_pixel_qres_t2_2way.html"
                        )
                except Exception as exc:
                    print(f"[WARNING] Pixel 2-way Q/T² plot failed for {config_id}: {exc!r}")

else:
    print("2-way visualization skipped.")


In [ ]:
# ---------------------------------------------------------------------
# Visual diagnostics requested by tutors
# ---------------------------------------------------------------------

def choose_diagnostic_configs(reference_df, n_per_family=2):
    group_cols = ["matrix_family"]

    if "candidate_source" in reference_df.columns:
        group_cols.append("candidate_source")

    return (
        reference_df
        .sort_values("frozen_reference_rank")
        .groupby(group_cols, group_keys=False, dropna=False)
        .head(int(n_per_family))
        .reset_index(drop=True)
    )


def choose_images_for_config_3way(config_id, n_images=3, mode="worst"):
    d = mixture_three_way_by_image_df[
        mixture_three_way_by_image_df["selected_config_id"].astype(str).eq(str(config_id))
    ].copy()

    if len(d) == 0:
        return []

    if mode == "best":
        d = d.sort_values(
            [
                "target_miss_rate",
                "non_target_false_accept_rate",
                "uncertain_rate",
            ],
            ascending=[True, True, True],
        )
    else:
        d = d.sort_values(
            [
                "target_miss_rate",
                "non_target_false_accept_rate",
                "uncertain_rate",
            ],
            ascending=[False, False, False],
        )

    return (
        d["source_image"]
        .astype(str)
        .drop_duplicates()
        .head(int(n_images))
        .tolist()
    )


diagnostic_configs_df = choose_diagnostic_configs(
    reference_configs_df,
    n_per_family=N_CONFIGS_TO_VISUALIZE_PER_FAMILY,
)

if RUN_VISUALIZATION:
    for _, cfg in diagnostic_configs_df.iterrows():
        config_id = str(cfg["selected_config_id"])

        obj_3way_df = mixture_objects_3way_df[
            mixture_objects_3way_df["selected_config_id"].astype(str).eq(config_id)
        ].copy()

        pix_3way_df = mixture_pixels_3way_df[
            mixture_pixels_3way_df["selected_config_id"].astype(str).eq(config_id)
        ].copy()

        image_keys = choose_images_for_config_3way(
            config_id=config_id,
            n_images=N_IMAGES_PER_CONFIG,
            mode=IMAGE_SELECTION_MODE,
        )

        print("=" * 100)
        print(
            f"{config_id} | {cfg.get('candidate_source', '')} | "
            f"{cfg.get('matrix_family', '')} | {cfg.get('training_matrix_id', '')} | "
            f"{cfg.get('preprocessing', '')} | {cfg.get('selected_rule_name', '')}"
        )
        print("Images:", image_keys)

        # -------------------------------------------------------------
        # Figure 1 — Confusion table 3-way with confidence
        # -------------------------------------------------------------
        if plot_three_way_confusion_heatmap is not None:
            conf_obj = mixture_object_3way_confusion_df[
                mixture_object_3way_confusion_df["selected_config_id"].astype(str).eq(config_id)
            ].copy()

            if len(conf_obj) > 0:
                fig = plot_three_way_confusion_heatmap(
                    conf_obj,
                    title=f"Object-level 3-way confusion — {config_id}",
                    show=True,
                )

                if SAVE_FIGURES_HTML and fig is not None:
                    fig.write_html(
                        FIGURES_DIR / f"{config_id}_object_3way_confusion.html"
                    )
            else:
                print(f"[INFO] No object 3-way confusion table for {config_id}.")

        # -------------------------------------------------------------
        # Figure 2 — Objectwise 3-way decision map
        # -------------------------------------------------------------
        for image_key in image_keys:
            if plot_object_decision_map is not None and len(obj_3way_df) > 0:
                fig = plot_object_decision_map(
                    image_db=image_db,
                    object_db=object_db,
                    results_df=obj_3way_df,
                    image_key=image_key,
                    decision_col="decision_3way",
                    decision_to_code={
                        NON_TARGET_LABEL: 1,
                        UNCERTAIN_LABEL_USED: 2,
                        TARGET_CLASS: 3,
                    },
                    code_to_name={
                        1: NON_TARGET_LABEL,
                        2: UNCERTAIN_LABEL_USED,
                        3: TARGET_CLASS,
                    },
                    title=f"Objectwise 3-way decision — {config_id} — {image_key}",
                    show=True,
                )

                if SAVE_FIGURES_HTML and fig is not None:
                    fig.write_html(
                        FIGURES_DIR / f"{config_id}_{image_key}_objectwise_3way_map.html"
                    )

            # Optional pixel-level view of the same objectwise decision.
            if plot_pixel_three_way_decision_overlay is not None and len(pix_3way_df) > 0:
                fig = plot_pixel_three_way_decision_overlay(
                    image_key=image_key,
                    image_db=image_db,
                    pixel_df=pix_3way_df,
                    decision_col="decision_3way",
                    title=f"Pixel view of objectwise 3-way decision — {config_id} — {image_key}",
                    show=True,
                )

                if SAVE_FIGURES_HTML and fig is not None:
                    fig.write_html(
                        FIGURES_DIR / f"{config_id}_{image_key}_pixel_3way_map.html"
                    )

        # -------------------------------------------------------------
        # Figure 3 — SIMCA Q residuals vs Hotelling T²
        # -------------------------------------------------------------
        if plot_simca_q_t2_dataframe is not None:
            # Object-level Q/T²
            if len(obj_3way_df) > 0:
                try:
                    fig = plot_simca_q_t2_dataframe(
                        obj_3way_df,
                        level="object",
                        label_col="decision_3way",
                        confidence_col="three_way_confidence",
                        title=f"Object-level SIMCA Q residuals vs Hotelling T² — {config_id}",
                        show=True,
                    )

                    if SAVE_FIGURES_HTML and fig is not None:
                        fig.write_html(
                            FIGURES_DIR / f"{config_id}_object_qres_t2.html"
                        )
                except Exception as exc:
                    print(f"[WARNING] Object Q/T² plot failed for {config_id}: {exc!r}")

            # Pixel-level Q/T², sampled for readability.
            if len(pix_3way_df) > 0:
                pix_plot_df = pix_3way_df.copy()

                if len(pix_plot_df) > MAX_PIXELS_FOR_QT2_PLOT:
                    pix_plot_df = (
                        pix_plot_df
                        .groupby("decision_3way", group_keys=False, dropna=False)
                        .apply(
                            lambda g: g.sample(
                                n=min(len(g), max(1, MAX_PIXELS_FOR_QT2_PLOT // 3)),
                                random_state=RANDOM_STATE,
                            )
                        )
                        .reset_index(drop=True)
                    )

                try:
                    fig = plot_simca_q_t2_dataframe(
                        pix_plot_df,
                        level="pixel",
                        label_col="decision_3way",
                        confidence_col="three_way_confidence",
                        title=f"Pixel-level SIMCA Q residuals vs Hotelling T² — {config_id}",
                        show=True,
                    )

                    if SAVE_FIGURES_HTML and fig is not None:
                        fig.write_html(
                            FIGURES_DIR / f"{config_id}_pixel_qres_t2.html"
                        )
                except Exception as exc:
                    print(f"[WARNING] Pixel Q/T² plot failed for {config_id}: {exc!r}")

else:
    print("Visualization skipped.")

## 11. Save protocol and inspect result files

In [ ]:
mixture_application_protocol_df = pd.DataFrame([{
    "db_h5_path": str(DB_H5_PATH),
    "results_04c_dir": str(RESULTS_04C_DIR),
    "results_dir": str(RESULTS_DIR),

    "wavelength_mode": WAVELENGTH_MODE,
    "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
    "results_tag": RESULTS_TAG,
    "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "n_active_bands": int(len(wavelengths)),

    "target_class": TARGET_CLASS,
    "non_target_label": NON_TARGET_LABEL,
    "uncertain_label": UNCERTAIN_LABEL_USED,
    "reference_classes_json": json.dumps(list(REFERENCE_CLASSES)),

    "frozen_reference_configs_path": str(FROZEN_REFERENCE_CONFIGS_PATH),
    "pure_test_metrics_path": str(PURE_TEST_METRICS_PATH),

    "mixture_final_train_batches_json": json.dumps(MIXTURE_FINAL_TRAIN_BATCHES),
    "mixture_final_train_filters_json": json.dumps(MIXTURE_FINAL_TRAIN_FILTERS, default=str),
    "mixture_filters_json": json.dumps(MIXTURE_FILTERS, default=str),

    "random_state": int(RANDOM_STATE),
    "replace_balanced_pixels": bool(REPLACE_BALANCED_PIXELS),
    "cv_n_splits": int(CV_N_SPLITS) if CV_N_SPLITS is not None else np.nan,
    "cv_group_col": CV_GROUP_COL,

    "apply_fixed_three_way": bool(APPLY_FIXED_THREE_WAY),
    "three_way_threshold_source": "04C_frozen_reference_configs",

    "run_border_diagnostic": bool(RUN_BORDER_DIAGNOSTIC),
    "border_diagnostic_widths_json": json.dumps(BORDER_DIAGNOSTIC_WIDTHS),

    "run_truth_dilation_sensitivity": bool(RUN_TRUTH_DILATION_SENSITIVITY),
    "truth_dilation_radii_json": json.dumps(TRUTH_DILATION_RADII),

    "run_visualization": bool(RUN_VISUALIZATION),
    "save_figures_html": bool(SAVE_FIGURES_HTML),
    "figures_dir": str(FIGURES_DIR),

    "save_mixture_pixel_tables": bool(SAVE_MIXTURE_PIXEL_TABLES),

    "n_reference_configs": int(len(reference_configs_df)),
    "n_mixture_metrics": int(len(mixture_metrics_df)),
    "n_mixture_objects": int(len(mixture_objects_df)),
    "n_mixture_pixels": int(len(mixture_pixels_df)),
    "n_mixture_refit_errors": int(len(mixture_refit_errors_df)),
    "n_mixture_three_way_metrics": int(len(mixture_three_way_metrics_df)),
    "n_mixture_three_way_by_image": int(len(mixture_three_way_by_image_df)),
    "n_mixture_objects_3way": int(len(mixture_objects_3way_df)),
    "n_mixture_pixels_3way": int(len(mixture_pixels_3way_df)),
    "n_object_3way_confusion_rows": int(len(mixture_object_3way_confusion_df)),
    "n_pixel_3way_confusion_rows": int(len(mixture_pixel_3way_confusion_df)),
    "n_border_diagnostic_rows": int(len(mixture_border_diagnostic_df)),
    "n_truth_dilation_sensitivity_rows": int(len(mixture_truth_dilation_sensitivity_df)),

    "mixture_metrics_path": str(MIXTURE_METRICS_PATH),
    "mixture_model_summary_path": str(MIXTURE_MODEL_SUMMARY_PATH),
    "mixture_object_predictions_path": str(MIXTURE_OBJECT_PREDICTIONS_PATH),
    "mixture_object_errors_by_image_path": str(MIXTURE_OBJECT_ERRORS_BY_IMAGE_PATH),
    "mixture_pixel_errors_by_image_path": str(MIXTURE_PIXEL_ERRORS_BY_IMAGE_PATH),
    "mixture_three_way_metrics_path": str(MIXTURE_THREE_WAY_METRICS_PATH),
    "mixture_three_way_by_image_path": str(MIXTURE_THREE_WAY_BY_IMAGE_PATH),
    "mixture_object_predictions_3way_path": str(MIXTURE_OBJECT_PREDICTIONS_3WAY_PATH),
    "mixture_pixel_predictions_3way_path": str(MIXTURE_PIXEL_PREDICTIONS_3WAY_PATH),
    "mixture_object_3way_confusion_path": str(MIXTURE_OBJECT_3WAY_CONFUSION_PATH),
    "mixture_pixel_3way_confusion_path": str(MIXTURE_PIXEL_3WAY_CONFUSION_PATH),
    "mixture_object_simca_diagnostics_path": str(MIXTURE_OBJECT_SIMCA_DIAGNOSTICS_PATH),
    "mixture_pixel_simca_diagnostics_path": str(MIXTURE_PIXEL_SIMCA_DIAGNOSTICS_PATH),
    "mixture_border_diagnostic_path": str(MIXTURE_BORDER_DIAGNOSTIC_PATH),
    "mixture_truth_dilation_sensitivity_path": str(MIXTURE_TRUTH_DILATION_SENSITIVITY_PATH),
    "mixture_application_protocol_path": str(MIXTURE_APPLICATION_PROTOCOL_PATH),
}])

save_parquet(
    mixture_application_protocol_df,
    MIXTURE_APPLICATION_PROTOCOL_PATH,
)

print("Saved mixture application protocol:")
print(MIXTURE_APPLICATION_PROTOCOL_PATH)

display(mixture_application_protocol_df)

In [ ]:
print("Result files:")
display(list_result_files(RESULTS_DIR))

## 12. Final check

In [ ]:
print("05_simca_mixture_application.ipynb completed.")
print()
print("Essential outputs:")
print(" -", MIXTURE_METRICS_PATH)
print(" -", MIXTURE_MODEL_SUMMARY_PATH)
print(" -", MIXTURE_OBJECT_PREDICTIONS_PATH)
print(" -", MIXTURE_OBJECT_ERRORS_BY_IMAGE_PATH)
print(" -", MIXTURE_PIXEL_ERRORS_BY_IMAGE_PATH)
print(" -", MIXTURE_THREE_WAY_METRICS_PATH)
print(" -", MIXTURE_THREE_WAY_BY_IMAGE_PATH)
print(" -", MIXTURE_OBJECT_PREDICTIONS_3WAY_PATH)
print(" -", MIXTURE_PIXEL_PREDICTIONS_3WAY_PATH)
print(" -", MIXTURE_OBJECT_3WAY_CONFUSION_PATH)
print(" -", MIXTURE_PIXEL_3WAY_CONFUSION_PATH)
print(" -", MIXTURE_OBJECT_SIMCA_DIAGNOSTICS_PATH)
print(" -", MIXTURE_PIXEL_SIMCA_DIAGNOSTICS_PATH)
print(" -", MIXTURE_BORDER_DIAGNOSTIC_PATH)
print(" -", MIXTURE_TRUTH_DILATION_SENSITIVITY_PATH)
print(" -", MIXTURE_APPLICATION_PROTOCOL_PATH)
print(" -", FIGURES_DIR)

if mixture_refit_errors_df is not None and len(mixture_refit_errors_df) > 0:
    print(" -", MIXTURE_REFIT_ERRORS_PATH)

if SAVE_MIXTURE_PIXEL_TABLES:
    print(" -", MIXTURE_PIXEL_PREDICTIONS_MINIMAL_PATH)

print()
print("Summary:")
print(f" - Wavelength mode: {WAVELENGTH_MODE}")
print(f" - Active bands: {len(wavelengths)}")
print(f" - Reference configs applied: {len(reference_configs_df)}")
print(f" - Mixture object predictions: {len(mixture_objects_df)}")
print(f" - Mixture pixel predictions in memory: {len(mixture_pixels_df)}")
print(f" - Mixture refit errors: {len(mixture_refit_errors_df)}")
print(f" - Fixed 3-way metric rows: {len(mixture_three_way_metrics_df)}")
print(f" - Fixed 3-way by-image rows: {len(mixture_three_way_by_image_df)}")
print(f" - Object 3-way confusion rows: {len(mixture_object_3way_confusion_df)}")
print(f" - Pixel 3-way confusion rows: {len(mixture_pixel_3way_confusion_df)}")
print(f" - Border diagnostic rows: {len(mixture_border_diagnostic_df)}")
print(f" - Truth dilation sensitivity rows: {len(mixture_truth_dilation_sensitivity_df)}")